# TheLook 이커머스 다차원 추천 및 공급망/지역 그래프 분석 실습

본 노트북은 BigQuery 상의 공개 데이터셋 `thelook_ecommerce`를 가공하여 **상품 연관 추천, 물류 공급망(Distribution Center), 고객의 지리적 위치(Location)** 정보를 하나로 연결한 다차원 속성 그래프(Property Graph)를 구축하고 GQL로 다중 홉(Multi-Hop) 비즈니스 패턴을 분석해 보는 통합 실습 워크북입니다.

---

## 💡 왜 BigQuery Property Graph 인가?
전통적인 관계형 데이터베이스(RDB) 환경에서 여러 단계를 거쳐 연결되는 다중 홉(Multi-Hop) 데이터 관계를 분석하는 것은 매우 어렵습니다. 

* **SQL의 한계 (다중 JOIN의 늪)**:
  * 고객이 구매한 상품 $\rightarrow$ 그 상품과 동시 구매된 다른 상품 $\rightarrow$ 해당 상품을 배송하는 물류 센터 $\rightarrow$ 고객이 사는 위치 등을 추적하려면 수많은 `JOIN` 연산이 필요합니다.
  * 테이블 수가 늘어날수록 SQL 쿼리는 기하급수적으로 복잡해지고 가독성이 떨어지며, 인덱싱이나 튜닝 없이는 대량 조인 연산 시 성능 저하가 발생합니다.
* **GQL (Graph Query Language)의 혁신**:
  * 속성 그래프(Property Graph) 모델을 적용하면 노드(Node, 데이터 객체)와 엣지(Edge, 관계선)로 현실 세계의 비즈니스 연결 관계를 직관적으로 선언할 수 있습니다.
  * 복잡한 조인 연산 없이 `MATCH (c:Customer)-[:ordered]->(p:Product)-[:supplied_from]->(dc)` 와 같은 화살표 기호 표현식을 사용해 직관적이고 빠르게 관계를 추적 및 쿼리할 수 있습니다.
  * BigQuery는 대용량 분산 스토리지 아키텍처 위에 GQL(SQL:2023 표준) 엔진을 내장하여, 별도의 그래프 DB를 구축하거나 데이터를 복제할 필요 없이 BigQuery 내부에서 초대형 그래프 연산을 즉시 수행할 수 있는 강력한 장점을 가집니다.

---

## 📂 데이터 레이어 분리 설계
*   **원천 데이터셋**: `thelook_ecommerce` (users, products, order_items, distribution_centers)
*   **정제 그래프 데이터셋**: `thelook_network` (실습을 통해 구축되는 가공 테이블 및 그래프)

---

### 학습 목표
1. **BigQuery Property Graph 모델링**: 노드(Node) 테이블과 엣지(Edge) 테이블을 설계하고 빅쿼리에 프로퍼티 그래프로 등록하는 기법을 배웁니다.
2. **GQL(Graph Query Language) 쿼리 작성**: SQL:2023 표준 GQL 구문(`MATCH`, `GRAPH_TABLE`)을 활용해 조인 없이 다중 홉 관계 데이터를 조회합니다.
3. **그래프 분석 시각화 및 최적화**: 시각화 쿼리와 `GRAPH_EXPAND` 패턴을 활용해 그래프 다차원 집계 성능을 최적화하는 기법을 실증합니다.

### 작업 파이프라인

1. **환경 초기화**: BigQuery Jupyter 매직 확장을 로드하고 정제된 그래프 데이터를 저장할 데이터셋을 생성합니다.
2. **그래프 노드(Node) 테이블 구축**: 원본 데이터를 가공하여 고객, 상품, 지리 위치, 물류 센터 정보가 담긴 노드 테이블을 생성합니다.
3. **그래프 관계(Edge) 테이블 구축**: 주문, 동시 구매, 거주 위치, 상품 공급 센터 간의 연관 관계를 나타내는 엣지 테이블을 생성합니다.
4. **Property Graph 정의 및 등록**: 구축된 노드와 엣지 테이블들의 조인 키 및 매핑 속성을 정의하여 BigQuery 내에 Property Graph를 등록합니다.
5. **GQL 다차원 분석 및 추천 실습**: 등록된 그래프에 대해 다양한 홉(Hop) 수의 GQL 매칭 쿼리를 실행하여 연관 상품 추천 및 지리적 공급망 흐름을 추적합니다.
6. **시각화 및 복합 관계 검증**: 특정 고객 중심의 하위 네트워크 그래프를 시각화하고 실무적인 복합 브랜드 관계 분석을 수행합니다.
7. **성능 및 아키텍처 비교 분석**: Naive SQL 방식과 GQL MATCH 방식, GRAPH_EXPAND 최적화 방식 간의 쿼리 가독성 및 비용 격차를 비교 분석합니다.

## Step 1: 환경 초기화

In [ ]:
# 1. BigQuery 확장을 활성화하고 정제 레이어용 데이터셋을 생성합니다.
%load_ext google.cloud.bigquery

from google.cloud import bigquery
from google.api_core.exceptions import Conflict

client = bigquery.Client()
dataset_id = f"{client.project}.thelook_network"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "asia-northeast3"  # 원천 데이터와 동일 리전 설정

try:
    client.create_dataset(dataset, timeout=30)
    print(f"Dataset '{dataset_id}' created successfully.")
except Conflict:
    print(f"Dataset '{dataset_id}' already exists.")

## Step 2: 그래프 노드(Node) 테이블 구축

원본 데이터로부터 **고객, 상품, 지리적 위치, 물류 센터**를 고유하게 식별할 수 있는 노드(Node) 성격의 마스터 테이블들을 가공하여 생성합니다. GQL 내에서 식별자로 활용될 기본 키(PK)의 중복 및 Null 값 검증을 동시에 확인합니다.

In [ ]:
%%bigquery
# 고객 노드 (node_users) 생성
CREATE OR REPLACE TABLE thelook_network.node_users AS
SELECT 
  id AS user_id,
  concat(first_name, ' ', last_name) AS user_name,
  age,
  gender,
  country,
  city,
  traffic_source
FROM thelook_ecommerce.users;

In [ ]:
%%bigquery
# 상품 노드 (node_products) 생성
CREATE OR REPLACE TABLE thelook_network.node_products AS
SELECT 
  id AS product_id,
  name AS product_name,
  category,
  brand,
  retail_price
FROM thelook_ecommerce.products;

In [ ]:
%%bigquery
# 고객 거주지 위치 노드 (node_locations) 생성
CREATE OR REPLACE TABLE thelook_network.node_locations AS
SELECT 
  ROW_NUMBER() OVER() AS location_id,
  country,
  city
FROM (
  SELECT DISTINCT country, city 
  FROM thelook_ecommerce.users 
  WHERE city IS NOT NULL AND country IS NOT NULL
);

In [ ]:
%%bigquery
# 물류 공급 센터 노드 (node_distribution_centers) 생성
CREATE OR REPLACE TABLE thelook_network.node_distribution_centers AS
SELECT 
  id AS center_id,
  name AS center_name,
  latitude,
  longitude
FROM thelook_ecommerce.distribution_centers;

## Step 3: 그래프 관계(Edge) 테이블 구축

노드 간 연결선인 **주문(`ordered`), 동시 구매(`co_purchased`), 위치 거주(`lives_in`), 상품 공급(`supplied_from`)** 테이블을 정의합니다. GQL 내부에서 관계선 홉(Hop)을 전개하기 위한 노드 테이블 간 조인 키 맵을 바탕으로 에지 데이터를 로드합니다.

In [ ]:
%%bigquery
# 주문 엣지 (edge_ordered) 생성
CREATE OR REPLACE TABLE thelook_network.edge_ordered AS
SELECT 
  id AS order_item_id,
  user_id,
  product_id,
  order_id,
  status,
  created_at,
  sale_price
FROM thelook_ecommerce.order_items;

In [ ]:
%%bigquery
# 동시구매 연관 엣지 (edge_co_purchased) 생성
CREATE OR REPLACE TABLE thelook_network.edge_co_purchased AS
WITH CoPurchases AS (
  SELECT 
    a.product_id AS source_product_id,
    b.product_id AS target_product_id,
    COUNT(DISTINCT a.order_id) AS co_purchase_count
  FROM thelook_ecommerce.order_items a
  JOIN thelook_ecommerce.order_items b 
    ON a.order_id = b.order_id 
    AND a.product_id < b.product_id
  GROUP BY source_product_id, target_product_id
)
SELECT 
  CONCAT(source_product_id, '_', target_product_id, '_copurchase') AS edge_id,
  source_product_id,
  target_product_id,
  co_purchase_count
FROM CoPurchases
WHERE co_purchase_count >= 2;

In [ ]:
%%bigquery
# 거주 관계 엣지 (edge_lives_in) 생성
CREATE OR REPLACE TABLE thelook_network.edge_lives_in AS
SELECT 
  CONCAT(u.user_id, '_lives_in_', l.location_id) AS edge_id,
  u.user_id,
  l.location_id
FROM thelook_network.node_users u
JOIN thelook_network.node_locations l
  ON u.country = l.country 
  AND u.city = l.city;

In [ ]:
%%bigquery
# 상품 공급망 엣지 (edge_supplied_from) 생성
CREATE OR REPLACE TABLE thelook_network.edge_supplied_from AS
SELECT 
  CONCAT(id, '_supplied_from_', distribution_center_id) AS edge_id,
  id AS product_id,
  distribution_center_id AS center_id
FROM thelook_ecommerce.products;

## Step 4: BigQuery Property Graph 등록

선언된 4개의 노드 테이블과 4개의 엣지 테이블들의 고유 키 및 관계 매핑 스펙을 통합 정의하여, BigQuery 메타데이터 상에 공식 속성 그래프(Property Graph) 리소스를 생성합니다. GQL은 이 스키마 정의를 참조하여 실시간 다중 조인을 해석하게 됩니다.

In [ ]:
%%bigquery
CREATE OR REPLACE PROPERTY GRAPH thelook_network.product_recommendation_graph
NODE TABLES (
  thelook_network.node_products AS Product KEY (product_id) 
    DEFAULT LABEL OPTIONS (description="Products sold in TheLook store", synonyms=["item", "merchandise"])
    PROPERTIES (
      product_id OPTIONS (description="Unique product ID", synonyms=["product code", "item id"]),
      category OPTIONS (description="Category of the product", synonyms=["product type", "product group"]),
      brand OPTIONS (description="Brand of the product", synonyms=["brand name", "manufacturer"]),
      product_name OPTIONS (description="Name of the product", synonyms=["title", "item name"]),
      retail_price OPTIONS (description="Retail price of the product", synonyms=["price", "tag price"]),
      MEASURE(AVG(retail_price)) AS avg_retail_price OPTIONS (description="Average retail price of products", synonyms=["average price"]),
      MEASURE(MAX(retail_price)) AS max_retail_price OPTIONS (description="Maximum retail price of products", synonyms=["max price"]),
      MEASURE(MIN(retail_price)) AS min_retail_price OPTIONS (description="Minimum retail price of products", synonyms=["min price"])
    ),
  thelook_network.node_users AS Customer KEY (user_id) 
    DEFAULT LABEL OPTIONS (description="Ecommerce customers who register or buy products", synonyms=["user", "shopper"])
    PROPERTIES (
      user_id OPTIONS (description="Unique customer user ID", synonyms=["user ID", "customer key"]),
      age OPTIONS (description="Age of the customer", synonyms=["customer age"]),
      gender OPTIONS (description="Gender of the customer", synonyms=["sex"]),
      traffic_source OPTIONS (description="Traffic source where the customer registered from", synonyms=["referral"]),
      MEASURE(AVG(age)) AS avg_customer_age OPTIONS (description="Average age of customers", synonyms=["average age"]),
      MEASURE(COUNT(user_id)) AS customer_count OPTIONS (description="Total count of customers", synonyms=["number of users", "user count"])
    ),
  thelook_network.node_locations AS Location KEY (location_id) 
    DEFAULT LABEL OPTIONS (description="Geographical locations of customers", synonyms=["address", "residence"])
    PROPERTIES (
      city OPTIONS (description="City name of the customer location", synonyms=["town"]),
      country OPTIONS (description="Country name of the customer location", synonyms=["nation"])
    ),
  thelook_network.node_distribution_centers AS DistributionCenter KEY (center_id) 
    DEFAULT LABEL OPTIONS (description="Logistics distribution centers supplying products", synonyms=["warehouse", "shipping center"])
    PROPERTIES (
      center_id OPTIONS (description="Unique ID of distribution center", synonyms=["center code", "warehouse ID"]),
      center_name OPTIONS (description="Name of the distribution center", synonyms=["warehouse name"])
    )
)
EDGE TABLES (
  thelook_network.edge_ordered AS ordered
    KEY (order_item_id)
    SOURCE KEY (user_id) REFERENCES Customer (user_id)
    DESTINATION KEY (product_id) REFERENCES Product (product_id)
    DEFAULT LABEL OPTIONS (description="Order transactions made by customers for products", synonyms=["purchased", "bought"])
    PROPERTIES (
      order_item_id OPTIONS (description="Unique order item ID", synonyms=["order item code"]),
      sale_price OPTIONS (description="The actual sale price of the ordered item", synonyms=["sold price", "checkout cost"]),
      status OPTIONS (description="Order status", synonyms=["delivery status", "order state"]),
      MEASURE(SUM(sale_price)) AS total_sales_amount OPTIONS (description="Total sales amount or revenue", synonyms=["revenue", "sales volume", "total spent"]),
      MEASURE(AVG(sale_price)) AS avg_sales_price OPTIONS (description="Average sale price of ordered items", synonyms=["average ticket size", "mean order value"]),
      MEASURE(COUNT(order_item_id)) AS order_count OPTIONS (description="Total count of ordered items", synonyms=["orders count", "number of items ordered"])
    ),
    
  thelook_network.edge_co_purchased AS co_purchased_with
    KEY (edge_id)
    SOURCE KEY (source_product_id) REFERENCES Product (product_id)
    DESTINATION KEY (target_product_id) REFERENCES Product (product_id)
    DEFAULT LABEL OPTIONS (description="Relationship between products bought together in the same order", synonyms=["co-purchased", "frequently bought together"])
    PROPERTIES (
      co_purchase_count OPTIONS (description="Number of times these two products were ordered together", synonyms=["co-purchase frequency"]),
      MEASURE(SUM(co_purchase_count)) AS total_co_purchase_count OPTIONS (description="Sum of co-purchase counts", synonyms=["total co-purchase frequency"])
    ),
    
  thelook_network.edge_lives_in AS lives_in
    KEY (edge_id)
    SOURCE KEY (user_id) REFERENCES Customer (user_id)
    DESTINATION KEY (location_id) REFERENCES Location (location_id)
    DEFAULT LABEL OPTIONS (description="Relationship indicating where a customer resides", synonyms=["resides", "located in"])
    NO PROPERTIES,
    
  thelook_network.edge_supplied_from AS supplied_from
    KEY (edge_id)
    SOURCE KEY (product_id) REFERENCES Product (product_id)
    DESTINATION KEY (center_id) REFERENCES DistributionCenter (center_id)
    DEFAULT LABEL OPTIONS (description="Relationship indicating which distribution center supplies a product", synonyms=["sourced from", "supplied by"])
    NO PROPERTIES
);

## Step 5: GQL 다차원 분석 및 추천 실습

---

### [기본 실습] 실습 0: 특정 고객(예: ID = 367)이 구매한 상품 목록 조회 (1-Hop Match)
가장 기초적인 1단계 홉 탐색으로, 특정 고객 노드에서 구매 엣지를 타고 상품 노드로 건너가 관련 정보를 쿼리하는 예시를 실습합니다.

In [ ]:
%%bigquery
SELECT 
  product_name,
  brand,
  category
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (c:Customer)-[:ordered]->(p:Product)
  WHERE c.user_id = 367
  RETURN p.product_name AS product_name, p.brand AS brand, p.category AS category
)
ORDER BY product_name;

In [ ]:
%%bigquery
SELECT center_name, COUNT(*) AS supply_count
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (l:Location)<-[:lives_in]-(c:Customer)-[:ordered]->(p:Product)-[:supplied_from]->(dc:DistributionCenter)
  WHERE l.country = 'Japan'
  RETURN dc.center_name AS center_name
)
GROUP BY center_name
ORDER BY supply_count DESC;

### 실습 2: 특정 물류 센터(예: ID = 1)가 서빙하는 주요 고객 거주 국가 Top 3 (4-Hop 탐색)

`DistributionCenter ➔ Product ➔ Customer ➔ Location` 관계 체인을 거꾸로 거슬러 올라가 물류 센터의 지리적 시장 영향도를 평가합니다.

In [ ]:
%%bigquery
SELECT country, COUNT(*) AS shipment_count
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (dc:DistributionCenter)<-[:supplied_from]-(p:Product)<-[:ordered]-(c:Customer)-[:lives_in]->(l:Location)
  WHERE dc.center_id = 1
  RETURN l.country AS country
)
GROUP BY country
ORDER BY shipment_count DESC
LIMIT 3;

## Step 6: 시각화 및 복합 관계 검증

특정 유저(`user_id = 367`)를 중심으로 구매한 상품들과 그 상품을 공급하는 물류 센터(Distribution Centers)까지의 전체 연결 경로를 BigQuery 시각화 매직 명령어(`--graph`)를 사용하여 2D 인터랙티브 그래프 형태로 즉시 렌더링하고 직관적으로 검증합니다.

In [ ]:
%%bigquery --graph display_only
SELECT
  TO_JSON(c) AS Customer_Node,
  TO_JSON(p) AS Product_Node,
  TO_JSON(o) AS Ordered_Edge,
  TO_JSON(dc) AS DC_Node,
  TO_JSON(sf) AS Supplied_Edge
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (c:Customer)-[o:ordered]->(p:Product)-[sf:supplied_from]->(dc:DistributionCenter)
  WHERE c.user_id = 367
  RETURN c, p, o, dc, sf
)
LIMIT 100;

### 실무형 5-Hop 복합 관계 분석 시나리오

그래프 데이터 모델의 진정한 강력함은 4홉(Hop) 이상의 복잡한 복합 관계를 단 한 줄의 화살표 관계식(`MATCH`)으로 기술할 수 있다는 데 있습니다. 

#### [시나리오 A] 연관 브랜드 크로스 추천 & 배송 공급망 매칭 (5-Hop Match)
"특정 고객이 구매한 브랜드(예: Calvin Klein)와 동일 브랜드를 구매한 다른 고객들(동반 구매 집단)을 찾고, 그들이 동시 구매한 다른 추천 카테고리의 상품 브랜드와 해당 상품이 출발하는 배송 물류 센터명까지의 5단계 경로 전체를 일괄 추적합니다."

In [ ]:
%%bigquery
SELECT 
  category AS recommended_category,
  brand AS recommended_brand,
  product_name AS recommended_product,
  center_name AS dc_name,
  SUM(co_purchase_count) AS recommendation_strength
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (l:Location)<-[:lives_in]-(c:Customer)-[:ordered]->(p1:Product)-[co:co_purchased_with]->(p2:Product)-[:supplied_from]->(dc:DistributionCenter)
  WHERE l.city = 'Seoul'
    AND p1.product_id != p2.product_id
  RETURN p2.category, p2.brand, p2.product_name, dc.center_name, co.co_purchase_count
)
GROUP BY recommended_category, recommended_brand, recommended_product, dc_name
ORDER BY recommendation_strength DESC
LIMIT 10;

### [시나리오 B] 특정 브랜드(예: Levi's) 구매 고객의 지역별 크로스 브랜드 소비 및 공급망 분석 (5-Hop)
* **목적:** 특정 인기 브랜드('Levi's') 제품을 구매한 고객들의 거주 국가(Location) 분포를 연결하고, 이들이 동시 구매(Co-purchase)하는 다른 연관 브랜드 상품과 이 연관 상품들의 원래 출고처(Distribution Center)를 결합하여 추적합니다.
* **활용:** 글로벌 크로스 브랜드 마케팅 전략을 수립하고, 공동 프로모션을 수행할 파트너 브랜드 및 연합 물류망 구축 타당성을 분석합니다.
* **경로:** `Location ➔ Customer ➔ Product1(Levi's) ➔ Product2(Associated) ➔ DistributionCenter`

#### 📊 SQL vs GQL 쿼리 가시성 및 복잡도 비교

##### 1) 기존 Standard SQL로 작성하는 경우 (총 8개 테이블 조인 필요)
```sql
SELECT 
  l.country AS customer_country,
  p2.brand AS associated_brand,
  dc.center_name AS supplier_dc,
  SUM(co.co_purchase_count) AS connection_strength
FROM `thelook_network.node_distribution_centers` dc
JOIN `thelook_network.edge_supplied_from` esf ON dc.center_id = esf.center_id
JOIN `thelook_network.node_products` p2 ON esf.product_id = p2.product_id
JOIN `thelook_network.edge_co_purchased` co ON p2.product_id = co.target_product_id
JOIN `thelook_network.node_products` p1 ON co.source_product_id = p1.product_id
JOIN `thelook_network.edge_ordered` eo ON p1.product_id = eo.product_id
JOIN `thelook_network.node_users` c ON eo.user_id = c.user_id
JOIN `thelook_network.edge_lives_in` el ON c.user_id = el.user_id
JOIN `thelook_network.node_locations` l ON el.location_id = l.location_id
WHERE p1.brand = 'Levi\'s'
  AND p1.product_id != p2.product_id
GROUP BY customer_country, associated_brand, supplier_dc
ORDER BY connection_strength DESC
LIMIT 10;
```

##### 2) GQL로 작성하는 경우
```sql
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (dc:DistributionCenter)<-[:supplied_from]-(p2:Product)<-[co:co_purchased_with]-(p1:Product)<-[:ordered]-(c:Customer)-[:lives_in]->(l:Location)
  ...
)
```

In [ ]:
%%bigquery
SELECT 
  country AS customer_country,
  brand AS associated_brand,
  center_name AS supplier_dc,
  SUM(co_purchase_count) AS connection_strength
FROM GRAPH_TABLE(
  thelook_network.product_recommendation_graph
  MATCH (dc:DistributionCenter)<-[:supplied_from]-(p2:Product)<-[co:co_purchased_with]-(p1:Product)<-[:ordered]-(c:Customer)-[:lives_in]->(l:Location)
  WHERE p1.brand = 'Levi\'s'
    AND p1.product_id != p2.product_id
  RETURN l.country, p2.brand, dc.center_name, co.co_purchase_count
)
GROUP BY customer_country, associated_brand, supplier_dc
ORDER BY connection_strength DESC
LIMIT 10;

## Step 7: 성능 및 아키텍처 비교 분석

BigQuery의 Graph Measures(예: 엣지 컬럼 직접 매핑) 구조와 **`GRAPH_EXPAND`** 특수 최적화 구문을 사용하면, 경로 홉 수의 확장에 따른 조인 연산 곱(Cartesian Product) 현상을 방어하고 메모리 사용량을 최소화할 수 있습니다. 

아래 실습을 통해 표준 GQL 방식과 성능 최적화(GRAPH_EXPAND) 방식 간의 문법 가독성 및 실제 쿼리 스캔 성능(비용) 격차를 최종 대조합니다.

In [ ]:
%%bigquery
-- 1. 집계 최적화를 위한 공급망 특화 그래프 정의
-- Edge 테이블의 KEY를 'product_id'로 지정하여, 상품당 하나의 물류센터만 매핑되는 다대일(N:1) 관계를 명시합니다.
CREATE OR REPLACE PROPERTY GRAPH thelook_network.product_supply_graph
NODE TABLES (
  thelook_network.node_products AS Product KEY (product_id)
    PROPERTIES (
      product_id,
      product_name,
      retail_price,
      MEASURE(AVG(retail_price)) AS avg_retail_price OPTIONS (description="상품 카탈로그 평균 단가")
    ),
  thelook_network.node_distribution_centers AS DistributionCenter KEY (center_id)
    PROPERTIES (
      center_id,
      center_name
    )
)
EDGE TABLES (
  thelook_network.edge_supplied_from AS supplied_from
    KEY (product_id) -- edge_id 대신 product_id를 Key로 사용하여 N:1 관계 보장
    SOURCE KEY (product_id) REFERENCES Product (product_id)
    DESTINATION KEY (center_id) REFERENCES DistributionCenter (center_id)
    NO PROPERTIES
);

### 세 가지 방식의 집계 비교 실습
동일한 물류센터별 상품 평균 가격 집계를 다음 3가지 방식으로 수행하여 코드의 복잡도와 결과값의 정확성을 확인합니다.

1. **Naive SQL**: `distribution_centers`와 `products`를 `order_items`에 단순 3중 조인하여 `AVG(p.retail_price)`를 구함 (조인 중복 발생으로 **틀린 결과** 출력).
2. **Correct SQL (CTE)**: 서브쿼리(또는 CTE)로 상품 테이블을 먼저 물류센터별로 평균 집계한 후 최종 조인 (코드가 길고 복잡하지만 **정확한 결과** 출력).
3. **Graph Expand SQL**: Property Graph 모델의 `GRAPH_EXPAND` TVF와 `AGG()` 함수를 활용 (코드 조인 없이 가장 간결하게 **정확한 결과** 출력).

In [ ]:
%%bigquery
WITH naive_sql AS (
  SELECT 
    dc.name AS center_name,
    ROUND(AVG(p.retail_price), 2) AS naive_avg
  FROM thelook_ecommerce.distribution_centers dc
  LEFT JOIN thelook_ecommerce.products p ON p.distribution_center_id = dc.id
  LEFT JOIN thelook_ecommerce.order_items oi ON oi.product_id = p.id
  GROUP BY dc.name
),
correct_sql AS (
  WITH dc_catalog_price AS (
    SELECT 
      distribution_center_id,
      AVG(retail_price) AS avg_retail_price
    FROM thelook_ecommerce.products
    GROUP BY distribution_center_id
  )
  SELECT 
    dc.name AS center_name,
    ROUND(cp.avg_retail_price, 2) AS correct_avg
  FROM thelook_ecommerce.distribution_centers dc
  LEFT JOIN dc_catalog_price cp ON cp.distribution_center_id = dc.id
),
graph_sql AS (
  SELECT
    DistributionCenter_center_name AS center_name,
    ROUND(AGG(Product_avg_retail_price), 2) AS graph_avg
  FROM GRAPH_EXPAND("thelook_network.product_supply_graph")
  GROUP BY DistributionCenter_center_name
)
SELECT 
  n.center_name AS center_name,
  n.naive_avg AS naive_sql_incorrect,
  c.correct_avg AS correct_sql_correct,
  g.graph_avg AS graph_tvf_correct
FROM naive_sql n
JOIN correct_sql c ON n.center_name = c.center_name
JOIN graph_sql g ON n.center_name = g.center_name
ORDER BY n.center_name;

### 💡 결과 분석 및 비즈니스 가치
위 실행 결과표에서 보듯, **Naive SQL** 방식은 주문이 잦은 인기 상품 위주로 가중치가 반영되어 왜곡된 단가 평균을 보여주는 반면, **Correct SQL**과 **Graph TVF** 방식은 동일한 올바른 카탈로그 평균값을 보여줍니다.

#### BigQuery Graph가 선사하는 이점:
1. **정합성 보장**: 복잡한 N:M 관계가 얽힌 상황에서도 분석가가 조인 관계의 깊이나 데이터 중복을 직접 골치 아프게 고민할 필요 없이, 엔진 차원에서 정확한 수학적 집계를 안전하게 수행해 줍니다.
2. **코드 간결화 (생산성)**: Correct SQL은 서브쿼리와 CTE를 활용해 집계를 분리하느라 쿼리가 길어지지만, Graph TVF 방식은 `GRAPH_EXPAND` 호출과 `AGG()` 함수 하나로 이를 완벽히 단축시킵니다.
3. **분석 생산성 극대화**: 비즈니스 메트릭(`MEASURE`)이 데이터 모델(Graph) 내에 한 번만 정의되면, BI 대시보드나 AI 에이전트가 데이터 모델 구조를 읽어 조인 없이 즉각 쿼리를 던질 수 있으므로 개발 속도와 편의성이 대폭 증대됩니다.